## 01b Experiment 1a: Category Membership - Items Task Analysis

In [1]:
# Setup
import pandas as pd
import csv
import matplotlib.pyplot as plt
import numpy as np
import string
from tabulate import tabulate
from collections import defaultdict
import random
import pickle
import seaborn as sns
import ast

In [21]:
# Attributes + Items Categories
# FURNITURE # WEAPON # VEHICLE # FRUIT # VEGETABLE # CLOTHING

# Only Items Categories:
# BIRD # CARPENTER'S TOOL # SPORT # TOY

#### Raw Data

In [22]:
# Importing data

items_df  = pd.read_csv('01_catmemexp_itemstask_allmodels_raw.csv', dtype=str, quoting=csv.QUOTE_ALL, encoding = "utf-8")
items_df.head()

,model,prompt,prompt_id,runtime,category,output
0,Qwen2.5-7B-Instruct,\r\n The procedure will be as follo...,6,1.716947794,Bird,"sparrow, eagle, pigeon, owl, hummingbird, flam..."
1,Qwen2.5-7B-Instruct,\r\n The procedure will be as follo...,6,1.12112999,Clothing,"shirt, pants, dress, jacket, skirt, shoes, soc..."
2,Qwen2.5-7B-Instruct,\r\n The procedure will be as follo...,6,1.506614923,Carpenter's tool,"hammer, saw, drill, chisel, wrench, screwdrive..."
3,Qwen2.5-7B-Instruct,\r\n The procedure will be as follo...,6,1.337905169,Weapon,"sword, gun, spear, knife, bow, axe, dagger, ma..."
4,Qwen2.5-7B-Instruct,\r\n The procedure will be as follo...,6,1.284659624,Vegetable,"lettuce, carrot, tomato, broccoli, cucumber, s..."


In [ ]:
# Counting NA outputs

# Items
na_count = items_df['output'].isna().sum()
print(f"\nNumber of NA items outputs: {na_count}")
na_values = items_df[items_df['output'].isna()]
models_na_counts = na_values['model'].value_counts()
print("NA items for each model:")
print(models_na_counts)




Number of NA items outputs: 0
NA items for each model:
Series([], Name: count, dtype: int64)


#### Processing Outputs 1

In [ ]:
# Processing Outputs

# eliminating concatenation rows
items_df = items_df[items_df.iloc[:, 0] != 'model']

# model names
items_df['model'] = items_df['model'].where(~items_df['model'].str.contains('/'), items_df['model'].str.split('/').str[1])
models = items_df['model'].unique()
print("Tested Models: \n", items_df['model'].unique(), "\n")

# automatic processing
def modify_string(value):
    if isinstance(value, str):
        
        value = value.replace("_", " ")
        
        elements = [elem.strip() for elem in value.split(',')]
        
        modified_elements = []
        for elem in elements:  
            if elem.endswith('s'): 
                modified_elements.append(elem[:-1].strip())
            else:
                modified_elements.append(elem.strip())
        
        modified_elements.append(elements[-1].strip()) 
        
        return ','.join(modified_elements)
    
    return value

items_df['output'] = items_df['output'].str.lower()
items_df['output'] = items_df['output'].str.rstrip(string.punctuation).str.strip()  
items_df['output'] = items_df['output'].apply(modify_string)

#items_df.to_csv('01_catmemexp_itemstask_allmodels_processed.csv',index=False, quoting=csv.QUOTE_ALL, encoding='utf-8')

Tested Models: 
 ['Qwen2.5-7B-Instruct' 'Qwen2.5-14B-Instruct' 'Qwen2.5-32B-Instruct'
 'Qwen2.5-72B-Instruct' 'Meta-Llama-3.1-8B-Instruct'
 'Meta-Llama-3.1-70B-Instruct' 'gemini-3-pro' 'deepseek-chat' 'gpt-5.2'] 



In [25]:
# Entries check
print("\nItems Task nr. Outputs:")
items_df_counts = items_df['model'].value_counts()
print(items_df_counts)


Items Task nr. Outputs:
model
Qwen2.5-7B-Instruct            300
Qwen2.5-14B-Instruct           300
Qwen2.5-32B-Instruct           300
Qwen2.5-72B-Instruct           300
Meta-Llama-3.1-8B-Instruct     300
Meta-Llama-3.1-70B-Instruct    300
gemini-3-pro                   300
deepseek-chat                  300
gpt-5.2                        300
Name: count, dtype: int64


In [26]:
# ... MANUAL PROCESSING ... #

#### Output Processing 2

In [ ]:
# Re-importing data after manual processing
items_df = pd.read_csv('01_catmemexp_itemstask_allmodels_processed.csv', dtype=str, quoting=csv.QUOTE_ALL, encoding='ISO-8859-1')
items_df.head()

In [43]:
# Entries check
print("\nItems Task nr. Outputs:")
items_df_counts = items_df['model'].value_counts()
print(items_df_counts)


Items Task nr. Outputs:
model
Qwen2.5-7B-Instruct            300
Qwen2.5-14B-Instruct           300
Qwen2.5-32B-Instruct           300
Qwen2.5-72B-Instruct           300
Meta-Llama-3.1-8B-Instruct     300
Meta-Llama-3.1-70B-Instruct    300
gemini-3-pro                   300
deepseek-chat                  300
gpt-5.2                        300
Name: count, dtype: int64


In [ ]:
# Converting output column to list of items
items_df['output_list'] = items_df['output'].apply(lambda x: x.split(',') if isinstance(x, str) else [])
items_df.head()

In [ ]:
# Collecting unique items for each category

# Collect unique generated items for each category
results = defaultdict(list)

for category, group in items_df.groupby('category'):
    unique_items = set()
    
    for output_list in group['output_list']:
        unique_items.update(output_list)
        
    for item in unique_items:
        if item:  # Skip empty items
            results[category].append(item)

# Creating Visualization Table
for category, items in results.items():
    letter_dict = defaultdict(list)

    items.sort()

    for item in items:
        first_letter = item[0].upper()  
        letter_dict[first_letter].append(item)

    distinct_letters = sorted(letter_dict.keys())
    table = []

    max_words = max(len(letter_dict[letter]) for letter in distinct_letters)

    for i in range(max_words):
        row = []
        for letter in distinct_letters:
            word = letter_dict[letter][i] if i < len(letter_dict[letter]) else ""
            row.append(word)
        table.append(row)

    # Print Table for each category
    print(f"\nTable for Category: {category}\n")
    print(tabulate(table, headers=distinct_letters, tablefmt="fancy_grid"))

    # Save Table for each category to .csv
    #df_table = pd.DataFrame(table, columns=distinct_letters)
    #csv_filename = f"01_catmemexp_uniqueitems_{category}.csv"
    #df_table.to_csv(csv_filename, index=False)

In [ ]:
# Second Automated Processing based on first post-processing results
# All second-step processed outputs are saved to a new column "outputs_list_processed"

"""
# Function to process the output list
def process_outputs(row, transformations):
    if row['category'] == "Furniture":
        processed_list = []
        for item in row['output_list']:
            if item in transformations:
                if transformations[item] is not None:
                    processed_list.append(transformations[item])
            else:
                processed_list.append(item)
        return processed_list
    else:
        return row['output_list']

"""
def process_outputs(row, transformations):
    processed_outputs = []
    for item in row['output_list']:  
        processed_item = transformations.get(item, item)  
        if processed_item is not None: 
            processed_outputs.append(processed_item)
    return processed_outputs        

In [ ]:
# Create Output List Processed: normalizations

# Dictionary of normalizations
transformations = {
    # FURNITURE
    "andiron": None,
    "ab": None,
    "arcco": "arccot",
    "ba": None,
    "bakerÃƒÂ¢Ã¢Â‚Â¬Ã¢Â„Â¢s rack": None,
    "ball sleeper sleeper": "ball sleeper", 
    "bergÃƒÂ¨re": None,
    "bird cage stand"
    "bergère": None,
    "bu": None,
    "jig": None,
    "love": None,
    "pre": None,
    "rockingæ¤…": None,
    "bar stool": "barstool",
    "bean bag": "beanbag",
    "bean bag chair": "beanbag",
    "beanbag chair": "beanbag",
    "bench seat": "bench",
    "book shelve": "bookshelf",
    "bokkshelve": "bookshelf",
    "chaise": "chaise longue",
    "chaise lounge": "chaise longue",
    "chiffonier": "chiffonnier",
    "cros": "cross",
    "crutche": "crutch",
    "dai": "dais",
    "davit": None,
    "deck chair": "deckchair",
    "directors chair": "director chair",
    "director's chair": "director chair",
    "ergonomics chair": "ergonomic chair",
    "exp":None,
    "filing cabinet": "file cabinet",
    "futon mattres": "futon mattress",
    "futon sleeper sleeper": "futon sleeper",
    "foundation property property": "foundation property",
    "glider chair": "glider",
    "harne":"harness",
    "hydraulic pres": "hydraulic press",
    "hypothesi":"hypothesis",
    "iconostasi":"iconostasis",
    "in/out tray":"in-out tray",
    "jig":"jigger",
    "judges' bench":"judge's bench",
    "high chair": "highchair",
    "love seat": "loveseat",
    "loveseat": "loveseat",
    "mattre": "mattress",
    "mattres": "mattress",
    "mattresse": "mattress",
    "patio furniture": "patio set",
    "pat":None,
    "reception counter": "reception",
    "reception desk": "reception",
    "recliner \ armchair": "recliner",
    "roll-top desk": "rolltop desk",
    "secretary": "secretaire",
    "secretary desk": "secretaire",
    "sectional sofa": "sectional",
    "shelve": "shelf",
    "shelf unit": "shelf",
    "shelving": "shelf",
    "shelving module": "shelf",
    "shelving unit": "shelf",
    "sofaçƒçš„": "sofa",
    "storage unit": "storage",
    "tatami mat": "tatami",
    "tv": "television",
    "theil's u": None,
    "thermo":None,
    "trunc":None,
    "trus":None,
    "tour bu": "tour bus",
    "settle settle settle settle canapÃƒÂ£Ã‚Â©": None,
    "vanity unit": "vanity",
    "wardrobe cabinet": "wardrobe",
    #WEAPON
    "auto-cannon": "autocannon",
    "ax": "axe",
    "baseball-bat": "baseball bat",
    "battle-ax": "battleaxe",
    "battle axe": "battleaxe",
    "biological agent": "biological weapon",
    "bomba": "bomb",
    "boomer": "boomerang",
    "brass-knuckle": "brass knuckle",
    "bu": None,
    "bat": "baseball bat",
    "dagger abbreviated": "dagger",
    "daggeråŽ¿äººæ°‘æ”¿åºœ projectile": None,
    "drill round": "drill",
    "estoc": None,
    "emp device": "emp", 
    "falæœ‰æ™‚å€™chion": "falchion",
    "flintlock pistol": "flintlock",
    "frying pan emp grenade": "frying pan,emp,grenade",
    "ga": None,
    "gla": None,
    "gladiu": "gladius",
    "grenade laÐ½Ð¸ÐºÐ°Ð¼Ð¸atched uchne": None,
    "handgun orificeless tk bomber": "handung,bomber",
    "improvised explosive device": "ied",
    "knuckle d nÃ¤rä¸»éƒ¨ç‚¹ã€‚": None,
    "laser pistol": "lasergun",
    "laser gun": "lasergun",
    "light saber": "lightsaber",
    "machine gun": "machinegun",
    "morning star": "morningstar",
    "mus ××•×ª×” ket": None,
    "mä¸¥é‡ace": None,
    "nerve ga": "nerve gas",
    "nunchaku": "nunchuck",
    "paintball-gun": "paintball gun",
    "pepper-spray": "pepper spray",
    "pipe-bomb": "pipe bomb",
    "plasma gun": "plasmagun",
    "ray gun": "raygun",
    "rocket-propelled grenadeÇ”É¾à¨—Ø˜æ·¹ð˜¦»Ä›jÅ¡Ã­ hp": "rocket-propelled grenade",
    "rocket launcher": "rocketlauncher",
    "sabre":"saber",
    "sawed-off crossed pins nuclear warhead": "sawed-off,crossed pins,nuclear warhead",
    "silenced sniper infrared disruptor": "silenced sniper,infrared disruptor",
    "spear gun": "speargun",
    "stun-gun": "stun gun",
    "submachine gun": "submachinegun",
    "tear ga": "tear gas",
    "trebuch": "trebuchet",
    "viru": "virus",
    "wakiz2097465ashi": None,
    "war hammer": "warhammer",
    "zweihÃ¤nder": "zweihander",
    "è‚¡æƒæŠ•èµ„ crossbow": "crossbow",
    "è‚¡": None,
    "éž": None,
    "éž­ç‚®(chinese firecracker)": "chinese firecracker",
    #VEHICLE
    "bo": None,
    "bu": "bus",
    "canoeipzig": "canoe",
    "circu": "circus",
    "compa": None,
    "consciou": "consciousness",
    "consciousne": "consciousness",
    "crutche": "crutches",
    "discu": None,
    "dog sled": "dog sled",
    "exodu": "exodus",
    "ferri": None,
    "feryyboat" : "ferry",
    "forkl": None,
    "fungu": "fungus",
    "ga": None,
    "geniu": "genius",
    "genu": "genus",
    "gla": "glass",
    "golf cart": "golfcart",
    "gra": "grass",
    "hand truck": "handtruck",
    "hoverboard ÑÑƒÐ±Ð¼Ð°Ñ€Ð¸Ð½Ð°": "hoverboard",
    "illne": "illness",
    "lawn mower": "lawnmower",
    "loudne": "loudness",
    "ma": "mass",
    "mattre": "mattress",
    "mech": "mechanic",
    "metropoli": "metropolis",
    "minibu": "minibus",
    "mistre": "mistress",
    "pa": "pass",
    "paddle boat": "paddleboat",
    "platypu": "platypus",
    "prosthesi": "prosthesis",
    "race car": "racecar",
    "rickshawå¤§ä¸minivan": "ricksaw,minivan",
    "shuttle bu": "shuttle bus",
    "skateboarding": "skateboard",
    "specie": "species",
    "tandem bicycle": "tandem bike",
    "tractor-trailer": "tractor trailer",
    "travoi": None,
    "walru": "walrus",
    "wellne": "wellness",
    "wilderne": "wilderness",
    # FRUIT
    "correct to kiwi)": None,
    "cupuaÃ§u": "cupuacu",
    "dragon fruit": "dragonfruit",
    "honeydew" : "honeydew melon",
    "icanian (correct should be icewan but likely meant iwan which doesn't exist" : "icanian",
    "inkç“œ(chinese for bitter melon)": None,
    "kiwi berry": "kiwiberry",
    "longan": "longanberry",
    "nectarine dimensionle": "nectarine",
    "passion fruit": "passionfruit",
    "persimmony": "persimmon",
    "physali": "physalis",
    "plum slovak grapefruit": "plum,grapefruit",
    "ugli fruit": "ugli fruit",
    "viÃ§ela(banter term for small plum": None,
    "viburnum": "viburnum berry",
    "vi catering fruit": "catering fruit",
    "xeniaæžœ(chinese for a": "xenia",
    "xigua(chinese for water": "xigua",
    "ylang-ylang fruit": "ylang fruit",
    "Ø³": None,
    "Ø³ØªØ®Ø¯Ù… ØªÙ„ÙŠÙØ²ÙŠÙˆÙ†iya": None,
    "ziz": "ziziphus",
    # VEGETABLE
    "asparagu": "asparagus",
    "beet": "beetroot",
    "beetrootæ²®ä¸§åœ°ä¸¢": "beetroot",
    "brussel": "brussels sprout",
    "butter lettuce": "butterhead lettuce",
    "carrageen mo": "carrageenan",
    "cayenne": "cayenne pepper",
    "celeryæŽ¨è¿›ä¼šã‚’ã‚ªãƒ³": "celery",
    "collard": "collard green",
    "enoki": "enoki mushroom",
    "fenn": "fennel",
    "fiddlehead": "fiddlehead fern",
    "green beanlevelä¸å¯ä»¥å®žçŽ°": "green bean",
    "hearts of palm": "heart of palm",
    "jalapeÃ±o": "jalapeno",
    "kaleç½®èº«äºŽä¼šè®®ç‚¹": "kale",
    "land cre": "land cress",
    "lemongra": "lemongrass",
    "como": None,
    "mushroom (culinary context)": "mushroom",
    "mushroomæ ‡é¢˜ä¸º": "mushroom",
    "mustard": "mustard green",
    "ok": None,
    "peaså…³é—­è®²åº§ç»„ç»‡è®ºæ–‡å…³é—­æ¸¸æˆ": "pea",
    "pimento": "pimiento",
    "portobello": "portobello mushroom",
    "potatoe": "potato",
    "quinoa green": "quinoa",
    "romaine": "romaine lettuce",
    "savoy cabbage": "savory cabbage",
    "serrano": "serrano pepper",
    "shiitake": "shiitake mushroom",
    "soy bean": "soybean",
    "sweet potatoe": "sweet potato",
    "sweet corn": "sweetcorn",
    "tiger nut": "tiger nut",
    "tomatoe": "tomato",
    "turn :up": None,
    "yamä¿¡æ¯é¢˜ç›®å»ºè®¾å·¥ç¨‹è®°å¾—åœ¨æ‹å–": None,
    "yuca": "yucca",
    "yu": "yu choy",
    "ç›˜": None,
    "ç›˜è–¯": None,
    # CLOTHING
    "-anorak": "anorak",
    "bathing suit": "bathingsuit",
    "bow tie": "bowtie",
    "cocktail dre": "cocktail dress",
    "brooche": "brooch",
    "clothes horse": "clothes horse",
    "dre": "dress",
    "dresse": "dress",
    "flip-flop": "flip flop",
    "legging": "leggings",
    "maxi dre": "maxidress",
    "maxidre": "maxidress",
    "ndromeå¯¹è±¡ unitard": "unitard",
    "neglige":"negligee",
    "pea coat": "peacoat",
    "pumps secureid braceletåŽŸã£ã± scarf": "pumps,bracelet,scarf",
    "rain boot":"rainboot",
    "scarve": "scarf",
    "seamstre": "seamstress",
    "snow pant": "snowpant",
    "sundre": "sundress",
    "sunglasse": "sunglasses",
    "tank top": "tanktop",
    "track suit": "tracksuit",
    "trench coat": "trenchcoat",
    "waist coat": "waistcoat",
    "watche": "watch",
    "wedding dre": "wedding dress",
    "[[position among letters]]bonnet": "bonnet",
    "[[space|boots]]": "space boots",
    "âˆ’ tie": "tie",
    "ä½†å¥¹other languages]]": None
}

# Apply & show processing
items_df['output_list_processed'] = items_df.apply(lambda row: process_outputs(row, transformations), axis=1)
items_df.head()

In [ ]:
# Create Output List Processed: remove duplicates from each list
items_df['output_list_processed'] = items_df['output_list_processed'].apply(lambda x: list(set(x)))
items_df.head()

In [ ]:
# Create Output List Processed Nonoise: Removing Items with frequency below 1

# Function to filter items based on frequency
def filter_items(group):

    group['output_list_processed'] = group['output_list_processed'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )


    exploded = group['output_list_processed'].explode()
    

    items_count = exploded.value_counts()
    

    filtered_items = items_count[items_count > 1].index.tolist()


    group['output_list_processed_nonoise'] = group['output_list_processed'].apply(
        lambda x: [item for item in x if item in filtered_items]
    )
    
    return group


items_df = items_df.groupby(['model', 'category'], group_keys=False).apply(filter_items)


items_df.reset_index(drop=True, inplace=True)

items_df.head()

"""
df_exploded = items_df.explode('output_list_processed')
count_df = df_exploded.groupby(['category', 'model', 'output_list_processed']).size().reset_index(name='count')

# Step 3: Filter Items
filtered_items = count_df[count_df['count'] > 1]

# Step 4: Create a New Column with Filtered Lists
items_df['output_list_processed_nonoise'] = items_df['output_list_processed'].apply(
    lambda x: [item for item in x if item in filtered_items['output_list_processed'].values])
"""

C:\Users\AS\AppData\Local\Temp\ipykernel_25084\2878605733.py:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  items_df = items_df.groupby(['model', 'category'], group_keys=False).apply(filter_items)


"\ndf_exploded = items_df.explode('output_list_processed')\ncount_df = df_exploded.groupby(['category', 'model', 'output_list_processed']).size().reset_index(name='count')\n\n# Step 3: Filter Items\nfiltered_items = count_df[count_df['count'] > 1]\n\n# Step 4: Create a New Column with Filtered Lists\nitems_df['output_list_processed_nonoise'] = items_df['output_list_processed'].apply(\n    lambda x: [item for item in x if item in filtered_items['output_list_processed'].values])\n"

In [ ]:
items_df

In [ ]:
# Last Processing CheckPoint: Re-generating Unique Items Table 

# Collect unique generated items for each category
results = defaultdict(list)

for category, group in items_df.groupby('category'):
    unique_items = set()
    
    for output_list in group['output_list_processed']:
        unique_items.update(output_list)
        
    for item in unique_items:
        if item:  
            results[category].append(item)

# Creating Visualization Table
for category, items in results.items():
    letter_dict = defaultdict(list)

    items.sort()

    for item in items:
        first_letter = item[0].upper()  
        letter_dict[first_letter].append(item)

    distinct_letters = sorted(letter_dict.keys())
    table = []

    max_words = max(len(letter_dict[letter]) for letter in distinct_letters)

    for i in range(max_words):
        row = []
        for letter in distinct_letters:
            word = letter_dict[letter][i] if i < len(letter_dict[letter]) else ""
            row.append(word)
        table.append(row)

    # Print Table for each category
    print(f"\nTable for Category: {category}\n")
    print(tabulate(table, headers=distinct_letters, tablefmt="fancy_grid"))

    # Save Table for each category to .csv
    df_table = pd.DataFrame(table, columns=distinct_letters)
    csv_filename = f"01_catmemexp_uniqueitems_{category}.csv"
    df_table.to_csv(csv_filename, index=False)

In [ ]:
items_df.head()

In [52]:
# Re-order output_list_processed
def reorder_column_processed(row):
    order_dict = {item: i for i, item in enumerate(row['output_list'])}
    return sorted(row['output_list_processed'], key=lambda x: order_dict.get(x, float('inf')))

items_df['output_list_processed'] = items_df.apply(reorder_column_processed, axis=1)

# Re-order output_list_processed_nonoise
def reorder_column_processedn(row):
    order_dict = {item: i for i, item in enumerate(row['output_list'])}
    return sorted(row['output_list_processed_nonoise'], key=lambda x: order_dict.get(x, float('inf')))

items_df['output_list_processed_nonoise'] = items_df.apply(reorder_column_processedn, axis=1)

In [ ]:
items_df.head()

In [ ]:
# Remove Unused Categories

#items_df  = pd.read_csv('01_catmemexp_itemstask_allmodels_processed.csv', dtype=str, quoting=csv.QUOTE_ALL, encoding = "utf-8")

# List of categories to remove
categories_to_remove = ["Toy", "Carpenter's tool", "Sport", "Bird"]

# Remove rows where the category is in the list
filtered_df = items_df [~items_df ['category'].isin(categories_to_remove)]

# Now, filtered_df contains the DataFrame with the specified categories removed
filtered_df.head()

In [ ]:
#filtered_df.to_csv('01_catmemexp_itemstask_allmodels_processed_FINAL.csv',index=False, quoting=csv.QUOTE_ALL, encoding='utf-8')